In [1]:
!pip install peft==0.14.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 6.9 MB/s eta 0:00:00


In [2]:
import peft
import transformers

print("peft:", peft.__version__)
print("transformers:", transformers.__version__)

peft: 0.14.0
transformers: 5.0.0


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

OPTIONS    = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP  = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
MODEL_PATH = "/kaggle/input/models/manjitbaishya001/bert-base-uncased-pytorch-only/pytorch/default/2"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_PATH)

Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [4]:
label_150 = LABEL_MAP[train.iloc[150]['answer']]
print(f"Q1 answer: {label_150}")

Q1 answer: 2


Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [5]:
row0       = train.iloc[0]
formatted  = str(row0['prompt']) + " [SEP] " + str(row0['B'])
print(f"Q2 answer: {len(formatted)}")

Q2 answer: 407


Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [6]:
row0     = train.iloc[0]
inputs   = [str(row0['prompt']) + " [SEP] " + str(row0[opt]) for opt in OPTIONS]

encoded  = tokenizer(
    inputs,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

input_ids = encoded['input_ids'].unsqueeze(0)
print(f"Q3 answer: {input_ids.shape[1]}")

Q3 answer: 5


Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [7]:
batch_inputs = []

for i in range(16):
    row = train.iloc[i]
    inputs = [str(row["prompt"]) + " [SEP] " + str(row[opt]) for opt in OPTIONS]

    encoded = tokenizer(
        inputs,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    batch_inputs.append(encoded["input_ids"])

input_ids = torch.stack(batch_inputs)

print(f"Q4 answer: {input_ids.numel()}")

Q4 answer: 10240


Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [8]:
model_mc = AutoModelForMultipleChoice.from_pretrained(MODEL_PATH)
model_mc.eval()

row0     = train.iloc[0]
inputs   = [str(row0['prompt']) + " [SEP] " + str(row0[opt]) for opt in OPTIONS]

encoded  = tokenizer(
    inputs,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

input_ids      = encoded['input_ids'].unsqueeze(0)
attention_mask = encoded['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs = model_mc(input_ids=input_ids, attention_mask=attention_mask)

print(f"Q5 answer: {outputs.logits.shape[1]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: /kaggle/input/models/manjitbaishya001/bert-base-uncased-pytorch-only/pytorch/default/2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training o

Q5 answer: 5


Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [9]:
label = torch.tensor([LABEL_MAP[train.iloc[0]['answer']]])

with torch.no_grad():
    outputs_with_loss = model_mc(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=label
    )

loss = outputs_with_loss.loss
print(f"Q6 answer: {loss.dim()}")

Q6 answer: 0


LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

model_mc2 = AutoModelForMultipleChoice.from_pretrained(MODEL_PATH)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(model_mc2, lora_config)

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print(f"Q7 answer: {trainable}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: /kaggle/input/models/manjitbaishya001/bert-base-uncased-pytorch-only/pytorch/default/2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training o

Q7 answer: 295681


Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [11]:
def preprocess(rows):
    all_input_ids      = []
    all_attention_mask = []
    all_labels         = []

    for i in range(len(rows['prompt'])):
        inputs = [str(rows['prompt'][i]) + " [SEP] " + str(rows[opt][i]) for opt in OPTIONS]
        encoded = tokenizer(
            inputs,
            padding='max_length',
            truncation=True,
            max_length=128,
        )
        all_input_ids.append(encoded['input_ids'])
        all_attention_mask.append(encoded['attention_mask'])
        all_labels.append(LABEL_MAP[rows['answer'][i]])

    return {
        'input_ids':      all_input_ids,
        'attention_mask': all_attention_mask,
        'labels':         all_labels
    }

first_100  = train.head(100)
hf_dataset = Dataset.from_pandas(first_100)
hf_dataset = hf_dataset.map(preprocess, batched=True, remove_columns=hf_dataset.column_names)

first_item = hf_dataset[0]
input_ids_shape = len(first_item['input_ids']), len(first_item['input_ids'][0])

print(f"Q8 answer: {input_ids_shape[0]}")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Q8 answer: 5


Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [12]:
model_mc3 = AutoModelForMultipleChoice.from_pretrained(MODEL_PATH)
lora_model3 = get_peft_model(model_mc3, LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
))

def preprocess_32(rows):
    all_input_ids, all_attention_mask, all_labels = [], [], []
    for i in range(len(rows['prompt'])):
        inputs = [str(rows['prompt'][i]) + " [SEP] " + str(rows[opt][i]) for opt in OPTIONS]
        encoded = tokenizer(inputs, padding='max_length', truncation=True, max_length=64)
        all_input_ids.append(encoded['input_ids'])
        all_attention_mask.append(encoded['attention_mask'])
        all_labels.append(LABEL_MAP[rows['answer'][i]])
    return {'input_ids': all_input_ids, 'attention_mask': all_attention_mask, 'labels': all_labels}

first_32   = train.head(32)
ds_32      = Dataset.from_pandas(first_32)
ds_32      = ds_32.map(preprocess_32, batched=True, remove_columns=ds_32.column_names)
ds_32.set_format(type='torch')

training_args = TrainingArguments(
    output_dir='./lora_output',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_steps=999,
    report_to='none'
)

trainer = Trainer(
    model=lora_model3,
    args=training_args,
    train_dataset=ds_32
)

train_result = trainer.train()
print(f"Q9 answer: {trainer.state.global_step}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: /kaggle/input/models/manjitbaishya001/bert-base-uncased-pytorch-only/pytorch/default/2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training o

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,3.161510
2,3.131437
3,3.165756
4,3.241981


Q9 answer: 4


Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [13]:
lora_model3.eval()

row0    = train.iloc[0]
inputs  = [str(row0['prompt']) + " [SEP] " + str(row0[opt]) for opt in OPTIONS]
encoded = tokenizer(
    inputs,
    padding='max_length',
    truncation=True,
    max_length=64,
    return_tensors='pt'
)

input_ids      = encoded['input_ids'].unsqueeze(0)
attention_mask = encoded['attention_mask'].unsqueeze(0)

device = next(lora_model3.parameters()).device

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

with torch.no_grad():
    outputs = lora_model3(input_ids=input_ids, attention_mask=attention_mask)

probs = torch.softmax(outputs.logits, dim=-1)

answer_q10 = round(probs[0][4].item(), 4)
print(f"Q10 answer: {answer_q10}")

Q10 answer: 0.1993
